# Section 1: Import libraries and load CSV dataset

In [1]:
# Import all libraries
import matplotlib as plt
import pandas as pd
import numpy as num
import seaborn as sns
import yfinance as yf

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Download data set
ticker = "AMD"
df = yf.download(
    ticker,
    start = "2018-06-01",
    end = "2026-06-01"
)

df.to_csv("amd_stock_price.csv", index=True)
print("CSV file saved successfully")

ModuleNotFoundError: No module named 'yfinance'

# Section 2: Data Cleaning & Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RAW_PATH = "amd_stock_price.csv"  # change this to match where the file sits on your machine

df = pd.read_csv(RAW_PATH, skiprows=[1, 2], header=0)
df.columns = ["Date", "Close", "High", "Low", "Open", "Volume"]
df.head()


print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.info()
df.isnull().sum().sort_values(ascending=False) #find the empty rows

#check for duplicate rows
duplicate_count = df.duplicated().sum()
print("Number of duplicated rows:", duplicate_count)

clean_df = df.copy()
clean_df.head(2)

#clean column names and convert types
clean_df.columns = clean_df.columns.str.strip().str.lower()
clean_df["date"] = pd.to_datetime(clean_df["date"], errors="coerce")
clean_df = clean_df.sort_values("date").reset_index(drop=True)
clean_df.dtypes

#deal with missing values
missing = clean_df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

numeric_cols = ["close", "high", "low", "open", "volume"]

for col in numeric_cols:
    if clean_df[col].isnull().any():
        median_val = clean_df[col].median()
        clean_df[col] = clean_df[col].fillna(median_val)
        print(f"Filled {col} missing values with median: {median_val:.2f}")

print("\nRemaining missing values:")
print(clean_df[numeric_cols].isnull().sum())

#handle outliers

clean_df["return"] = clean_df["close"].pct_change()

q_low, q_high = clean_df["return"].quantile([0.001, 0.999])
n_extreme = ((clean_df["return"] < q_low) | (clean_df["return"] > q_high)).sum()
print(f"Extreme daily return outliers found: {n_extreme}")

clean_df["return"] = clean_df["return"].clip(q_low, q_high)

#create new columns for cleaner
for lag in [1, 2, 3, 5]:
    clean_df[f"close_lag{lag}"] = clean_df["close"].shift(lag)

clean_df["sma_5"] = clean_df["close"].shift(1).rolling(window=5).mean()
clean_df["sma_10"] = clean_df["close"].shift(1).rolling(window=10).mean()
clean_df["volatility_5"] = clean_df["return"].shift(1).rolling(window=5).std()
clean_df["volume_lag1"] = clean_df["volume"].shift(1)
clean_df["volume_sma_5"] = clean_df["volume"].shift(1).rolling(window=5).mean()
clean_df["hl_range_lag1"] = clean_df["high"].shift(1) - clean_df["low"].shift(1)

# Target: next trading day's closing price
clean_df["target"] = clean_df["close"].shift(-1)

clean_df.tail()

#remove structural missing values
before = len(clean_df)
clean_df = clean_df.dropna().reset_index(drop=True)
after = len(clean_df)
print(f"Dropped {before - after} rows with structural NaNs (feature warm-up period + final row with no target)")
print(f"Remaining rows: {after}")

clean_df.to_csv("amd_cleaned.csv", index=False)
print("Saved cleaned file: amd_cleaned.csv")

# Section 3: Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

df = pd.read_csv("amd_cleaned.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
df_feat = df.copy()

# RSI (Relative Strength Index) - measures market momentum
def compute_rsi(series, window=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df_feat["rsi_14"] = compute_rsi(df_feat["close"].shift(1), window=14)

# EMA - tracks price trend
df_feat["ema_12"] = df_feat["close"].shift(1).ewm(span=12, adjust=False).mean()
df_feat["ema_26"] = df_feat["close"].shift(1).ewm(span=26, adjust=False).mean()

# MACD - trend momentum
df_feat["macd"] = df_feat["ema_12"] - df_feat["ema_26"]
df_feat["macd_signal"] = df_feat["macd"].ewm(span=9, adjust=False).mean()
df_feat["macd_hist"] = df_feat["macd"] - df_feat["macd_signal"]

# Bollinger Bands - check volatility of price, in relation with recent changes
bb_window = 20
bb_mid = df_feat["close"].shift(1).rolling(bb_window).mean()
bb_std = df_feat["close"].shift(1).rolling(bb_window).std()
df_feat["bb_upper"] = bb_mid + 2 * bb_std
df_feat["bb_lower"] = bb_mid - 2 * bb_std
df_feat["bb_width"] = df_feat["bb_upper"] - df_feat["bb_lower"]
df_feat["bb_pct_b"] = (df_feat["close"].shift(1) - df_feat["bb_lower"]) / df_feat["bb_width"]

# ATR (Average True Range) - measures volatility of stock 
prev_close = df_feat["close"].shift(1)
high_low = df_feat["high"].shift(1) - df_feat["low"].shift(1)
high_prevclose = (df_feat["high"].shift(1) - prev_close).abs()
low_prevclose = (df_feat["low"].shift(1) - prev_close).abs()
true_range = pd.concat([high_low, high_prevclose, low_prevclose], axis=1).max(axis=1)
df_feat["atr_14"] = true_range.rolling(14).mean()

# HLC / OC Ratios - measures movement of stock throughout one day
df_feat["hlc_ratio"] = (df_feat["high"].shift(1) - df_feat["low"].shift(1)) / df_feat["close"].shift(1)
df_feat["oc_ratio"] = (df_feat["open"].shift(1) - df_feat["close"].shift(1)) / df_feat["close"].shift(1)

# ROC / Momentum - checks rate price changes
df_feat["roc_5"] = df_feat["close"].shift(1).pct_change(5)
df_feat["roc_10"] = df_feat["close"].shift(1).pct_change(10)
df_feat["momentum_5"] = df_feat["close"].shift(1) - df_feat["close"].shift(6)

# Ratio features - price/volume relative to their recent averages
df_feat["close_to_sma5"] = df_feat["close"].shift(1) / df_feat["sma_5"] - 1
df_feat["close_to_sma10"] = df_feat["close"].shift(1) / df_feat["sma_10"] - 1
df_feat["volume_to_volsma5"] = df_feat["volume"].shift(1) / df_feat["volume_sma_5"] - 1

# Return distribution
df_feat["return_skew_10"] = df_feat["return"].shift(1).rolling(10).skew()
df_feat["return_kurt_10"] = df_feat["return"].shift(1).rolling(10).kurt()

# Rolling min/max - looks at recent suport/resistance levels
df_feat["rolling_max_10"] = df_feat["high"].shift(1).rolling(10).max()
df_feat["rolling_min_10"] = df_feat["low"].shift(1).rolling(10).min()

# Calendar features
df_feat["day_of_week"] = df_feat["date"].dt.dayofweek
df_feat["month"] = df_feat["date"].dt.month
df_feat["is_monday"] = (df_feat["day_of_week"] == 0).astype(int)
df_feat["is_month_end"] = df_feat["date"].dt.is_month_end.astype(int)

# Drop NaNs
before = len(df_feat)
df_feat = df_feat.dropna().reset_index(drop=True)
print(f"Dropped {before - len(df_feat)} rows due to feature warm-up periods")
print(f"Remaining rows: {len(df_feat)}")

# Correlation check
corr_with_target = df_feat.corr(numeric_only=True)["target"].sort_values(ascending=False)
print("\nTop correlated features with target:")
print(corr_with_target.head(15))

df_feat.to_csv("amd_features.csv", index=False)
print("Saved amd_features.csv")

# TEST / TRAIN / SPLIT
train_df, test_df = train_test_split(df_feat, test_size=0.2, shuffle=False)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\nTrain: {train_df['date'].min()} to {train_df['date'].max()} ({len(train_df)} rows)")
print(f"Test:  {test_df['date'].min()} to {test_df['date'].max()} ({len(test_df)} rows)")

# ENCODING
categorical_cols = ["day_of_week", "month"]

train_df = pd.get_dummies(train_df, columns=categorical_cols, prefix=categorical_cols)
test_df = pd.get_dummies(test_df, columns=categorical_cols, prefix=categorical_cols)

train_cols = train_df.columns
train_df, test_df = train_df.align(test_df, join="left", axis=1, fill_value=0)
test_df = test_df[train_cols]

# FEATURE SCALING
no_scale_cols = ["date", "target", "is_monday", "is_month_end"]
no_scale_cols += [c for c in train_df.columns if c.startswith("day_of_week_") or c.startswith("month_")]
feature_cols = [c for c in train_df.columns if c not in no_scale_cols]

print(f"\nScaling {len(feature_cols)} numeric features")

scaler = StandardScaler()
train_scaled = train_df.copy()
test_scaled = test_df.copy()

train_scaled[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_scaled[feature_cols] = scaler.transform(test_df[feature_cols])

# Save df
train_scaled.to_csv("amd_train_scaled.csv", index=False)
test_scaled.to_csv("amd_test_scaled.csv", index=False)
print("Saved train file: amd_train_scaled.csv")
print("Saved test file: amd_test_scaled.csv")



# Section 4: Model Training

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ---------------------------------------------------------
# Step 0: Load train/test sets built in Section 3
# ---------------------------------------------------------
train_df = pd.read_csv("amd_train_scaled.csv", parse_dates=["date"])
test_df  = pd.read_csv("amd_test_scaled.csv", parse_dates=["date"])

feature_cols = [c for c in train_df.columns if c not in ("date", "target")]
X_train, y_train = train_df[feature_cols], train_df["target"]
X_test, y_test   = test_df[feature_cols], test_df["target"]

# Leakage-safe CV splitter reused for both models' tuning
tscv = TimeSeriesSplit(n_splits=5)

# ===========================================================
# ALGORITHM 1: Linear Regression
# ===========================================================

# Step 1: Baseline fit (no hyperparameters to tune - closed-form OLS solution)
lr = LinearRegression()
lr.fit(X_train, y_train)

# Step 2: Predict
lr_pred = lr.predict(X_test)

# Step 3: Loss function check - Linear Regression minimizes MSE (sum of squared
# residuals) during training. We report it on the held-out test set here.
lr_mse = mean_squared_error(y_test, lr_pred)
print(f"[Linear Regression] Test MSE: {lr_mse:.4f}")

# ===========================================================
# ALGORITHM 2: Random Forest Regressor
# ===========================================================

# Step 1: Define the hyperparameter grid to search
rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# Step 2: Grid Search - tries every combination above, scored with TimeSeriesSplit
# CV so no future data leaks into any validation fold. Loss function used
# internally by each tree is MSE (variance reduction on splits); scoring here
# is negative MSE so GridSearchCV can maximize it.
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=rf_param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

# Step 3: Fit - this runs the full search over the grid
rf_grid.fit(X_train, y_train)

# Step 4: Pull out the best model found by the search
rf_best = rf_grid.best_estimator_
print("[Random Forest] Best params:", rf_grid.best_params_)

# Step 5: Predict with the tuned model
rf_pred = rf_best.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_pred)
print(f"[Random Forest] Test MSE: {rf_mse:.4f}")


# Section 5: Evaluation

In [4]:
import os
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
import matplotlib.pyplot as plt

# --- Check the required files exist before doing anything else ---
required_files = ["amd_train_scaled.csv", "amd_test_scaled.csv"]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    print("Current working directory:", os.getcwd())
    print("Files found here:", os.listdir())
    raise FileNotFoundError(
        f"Missing: {missing}. These are generated by the Feature Engineering "
        f"section (Section 3) — make sure that cell has been run, or ask your "
        f"teammate for these CSVs and place them in the folder printed above."
    )

# --- Self-contained load (doesn't depend on earlier cells having run) ---
train_df = pd.read_csv("amd_train_scaled.csv", parse_dates=["date"])
test_df  = pd.read_csv("amd_test_scaled.csv", parse_dates=["date"])

feature_cols = [c for c in train_df.columns if c not in ("date", "target")]
X_train, y_train = train_df[feature_cols], train_df["target"]
X_test, y_test   = test_df[feature_cols], test_df["target"]

tscv = TimeSeriesSplit(n_splits=5)

lr = LinearRegression().fit(X_train, y_train)
rf_best = RandomForestRegressor(n_estimators=200, random_state=42).fit(X_train, y_train)

lr_pred, rf_pred = lr.predict(X_test), rf_best.predict(X_test)
lr_train_pred, rf_train_pred = lr.predict(X_train), rf_best.predict(X_train)

# --- Metrics table ---
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE":  [mean_absolute_error(y_test, lr_pred), mean_absolute_error(y_test, rf_pred)],
    "MSE":  [mean_squared_error(y_test, lr_pred),  mean_squared_error(y_test, rf_pred)],
    "RMSE": [mean_squared_error(y_test, lr_pred) ** 0.5, mean_squared_error(y_test, rf_pred) ** 0.5],
    "R2":   [r2_score(y_test, lr_pred), r2_score(y_test, rf_pred)]
})
print(results)

# Feature importance (RF)
importances = pd.Series(rf_best.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop 10 RF feature importances:")
print(importances.head(10))

# --- Overfitting check: train vs validation(CV) vs test ---
lr_cv_mse = -cross_val_score(lr, X_train, y_train, cv=tscv, scoring="neg_mean_squared_error").mean()
rf_cv_mse = -cross_val_score(rf_best, X_train, y_train, cv=tscv, scoring="neg_mean_squared_error").mean()

overfit_check = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "Train MSE": [mean_squared_error(y_train, lr_train_pred), mean_squared_error(y_train, rf_train_pred)],
    "Val (CV) MSE": [lr_cv_mse, rf_cv_mse],
    "Test MSE": [mean_squared_error(y_test, lr_pred), mean_squared_error(y_test, rf_pred)],
})
print("\nOverfitting check (Train vs Validation vs Test):")
print(overfit_check)

               Model        MAE          MSE       RMSE        R2
0  Linear Regression   5.343576    85.926500   9.269655  0.986980
1      Random Forest  25.750010  3824.712764  61.844262  0.420454

Top 10 RF feature importances:
close             0.794362
low               0.083266
high              0.047186
ema_26            0.028785
ema_12            0.017956
open              0.012011
close_lag1        0.004389
sma_5             0.002267
close_lag2        0.001576
rolling_max_10    0.001279
dtype: float64

Overfitting check (Train vs Validation vs Test):
               Model  Train MSE  Val (CV) MSE     Test MSE
0  Linear Regression   8.878664     12.621086    85.926500
1      Random Forest   1.519439    231.569925  3824.712764
